# Pipeline Machine Learning End-to-End dengan TensorFlow Extended (TFX) dan Apache Beam

Notebook ini mengimplementasikan Machine Learning Pipeline end-to-end berstandar produksi untuk estimasi harga rumah (*House Price Prediction*) menggunakan **TensorFlow Extended (TFX)** yang diorkestrasi secara otomatis dengan **Apache Beam (`BeamDagRunner`)**.

Pipeline mencakup 10 komponen terintegrasi:
1. **Validasi & Skema Data Otomatis:** Mengidentifikasi statistik dan anomali data.
2. **Preprocessing Konsisten:** Mencegah *train-serving skew* dengan TensorFlow Transform (`TFT`).
3. **Hyperparameter Tuning Otomatis:** Optimasi arsitektur jaringan menggunakan KerasTuner.
4. **Model Training & Evaluation:** Pelatihan Deep Neural Network dan validasi kualitas berbasis ambang batas (*threshold blessing*) menggunakan TFMA.
5. **Model Pusher:** Otomasi ekspor model *blessed* siap deploy ke direktori TensorFlow Serving.

## Inisialisasi dan Konfigurasi Pipeline

In [1]:
import os
import tensorflow as tf
from tfx.orchestration import metadata, pipeline
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner

2026-09-13 23:32:59.948643: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
USERNAME = 'erlanggajuni45'
PIPELINE_NAME = f'{USERNAME}-pipeline'

DATA_ROOT = os.path.join(PIPELINE_NAME, 'data')
PIPELINE_ROOT = os.path.join(PIPELINE_NAME, 'pipeline_root')
METADATA_DIR = os.path.join(PIPELINE_NAME, 'metadata')
METADATA_PATH = os.path.join(METADATA_DIR, 'metadata.db')
SERVING_MODEL_DIR = os.path.join(PIPELINE_NAME, 'serving_model')

TRANSFORM_MODULE_FILE = 'modules/transform.py'
TUNER_MODULE_FILE = 'modules/tuner.py'
TRAINER_MODULE_FILE = 'modules/trainer.py'

In [3]:
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)
os.makedirs(SERVING_MODEL_DIR, exist_ok=True)

## Perancangan Komponen Pipeline End-to-End

Fungsi `init_pipeline` merangkai 10 komponen TFX ke dalam sebuah *Directed Acyclic Graph* (DAG) yang akan dieksekusi oleh Apache Beam:

1. **CsvExampleGen:** Membaca dataset mentah berformat CSV dan membaginya menjadi partisi data latih (*train*) dan evaluasi (*eval*) dalam format standar TFRecord.
2. **StatisticsGen:** Menghitung ringkasan statistik deskriptif dari dataset untuk analisis karakteristik fitur numerik dan kategorikal.
3. **SchemaGen:** Menginferensi skema data secara otomatis, termasuk deteksi tipe data, keberadaan fitur, dan rentang nilai yang diharapkan.
4. **ExampleValidator:** Memvalidasi dataset terhadap skema yang dibuat untuk mendeteksi anomali data, *missing values*, atau ketidaksesuaian tipe fitur.
5. **Transform:** Melakukan rekayasa fitur (*feature engineering*) menggunakan TensorFlow Transform (`modules/transform.py`). Fitur numerik distandarisasi (*z-score*) dan fitur kategorikal dipetakan ke *vocabulary index* untuk menghindari *train-serving skew*.
6. **Tuner (Saran 1):** Mengoptimasi hyperparameter arsitektur model secara otomatis menggunakan KerasTuner (`modules/tuner.py`) dengan strategi *RandomSearch* untuk menemukan kombinasi *units*, *dropout rate*, dan *learning rate* terbaik.
7. **Trainer:** Melatih arsitektur Keras Deep Neural Network (`modules/trainer.py`) menggunakan hyperparameter terbaik hasil dari komponen Tuner serta mengekspor model lengkap beserta *serving signature* (`serve_tf_examples_fn`).
8. **Resolver (LatestBlessedModelStrategy):** Mengidentifikasi dan mengambil model baseline terbaik sebelumnya yang berstatus *blessed* sebagai pembanding performa.
9. **Evaluator:** Mengevaluasi performa model menggunakan TensorFlow Model Analysis (TFMA) pada metrik Mean Absolute Error (MAE) dan Mean Squared Error (MSE), serta menetapkan status *blessing* jika lolos ambang batas validasi (MSE $\le 10^{14}$).
10. **Pusher:** Menerima model yang telah disetujui (*blessed*) oleh Evaluator dan menyimpannya secara otomatis ke direktori deployment (`serving_model_dir`) untuk disajikan oleh TensorFlow Serving.

In [4]:
import tensorflow_model_analysis as tfma
from tfx.components import (
    CsvExampleGen,
    Evaluator,
    ExampleValidator,
    Pusher,
    SchemaGen,
    StatisticsGen,
    Trainer,
    Transform,
    Tuner,
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.proto import pusher_pb2, trainer_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

In [5]:
def init_pipeline(
    pipeline_root: str,
    pipeline_name: str,
    metadata_path: str,
    data_root: str,
    transform_module_file: str,
    tuner_module_file: str,
    trainer_module_file: str,
    serving_model_dir: str,
) -> pipeline.Pipeline:
  """Membangun dan merangkai 10 komponen Machine Learning Pipeline TFX."""

  # 1. Data Ingestion
  example_gen = CsvExampleGen(input_base=data_root)

  # 2. Ringkasan Statistik
  statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])

  # 3. Inferensi Skema
  schema_gen = SchemaGen(
      statistics=statistics_gen.outputs['statistics'], infer_feature_shape=True
  )

  # 4. Validasi Anomali Data
  example_validator = ExampleValidator(
      statistics=statistics_gen.outputs['statistics'],
      schema=schema_gen.outputs['schema'],
  )

  # 5. Rekayasa Fitur (Transform)
  transform = Transform(
      examples=example_gen.outputs['examples'],
      schema=schema_gen.outputs['schema'],
      module_file=transform_module_file,
  )

  # 6. Hyperparameter Tuning (Saran 1)
  tuner = Tuner(
      module_file=tuner_module_file,
      examples=transform.outputs['transformed_examples'],
      transform_graph=transform.outputs['transform_graph'],
      schema=schema_gen.outputs['schema'],
      train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=40),
      eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=20),
  )

  # 7. Pelatihan Model (Trainer)
  trainer = Trainer(
      module_file=trainer_module_file,
      examples=transform.outputs['transformed_examples'],
      transform_graph=transform.outputs['transform_graph'],
      schema=schema_gen.outputs['schema'],
      hyperparameters=tuner.outputs['best_hyperparameters'],
      train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=100),
      eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=50),
  )

  # 8. Resolver Model Baseline
  model_resolver = Resolver(
      strategy_class=LatestBlessedModelStrategy,
      model=Channel(type=Model),
      model_blessing=Channel(type=ModelBlessing),
  ).with_id('latest_blessed_model_resolver')

  # 9. Evaluasi & Validasi Model (Evaluator)
  eval_config = tfma.EvalConfig(
      model_specs=[tfma.ModelSpec(label_key='price')],
      slicing_specs=[tfma.SlicingSpec()],
      metrics_specs=[
          tfma.MetricsSpec(
              metrics=[
                  tfma.MetricConfig(class_name='MeanAbsoluteError'),
                  tfma.MetricConfig(
                      class_name='MeanSquaredError',
                      threshold=tfma.MetricThreshold(
                          value_threshold=tfma.GenericValueThreshold(
                              upper_bound={'value': 1e14}
                          )
                      ),
                  ),
              ]
          )
      ],
  )

  evaluator = Evaluator(
      examples=example_gen.outputs['examples'],
      model=trainer.outputs['model'],
      baseline_model=model_resolver.outputs['model'],
      eval_config=eval_config,
  )

  # 10. Deployment Model (Pusher)
  pusher = Pusher(
      model=trainer.outputs['model'],
      model_blessing=evaluator.outputs['blessing'],
      push_destination=pusher_pb2.PushDestination(
          filesystem=pusher_pb2.PushDestination.Filesystem(
              base_directory=serving_model_dir
          )
      ),
  )

  components = (
      example_gen,
      statistics_gen,
      schema_gen,
      example_validator,
      transform,
      tuner,
      trainer,
      model_resolver,
      evaluator,
      pusher,
  )

  return pipeline.Pipeline(
      pipeline_name=pipeline_name,
      pipeline_root=pipeline_root,
      components=components,
      metadata_connection_config=metadata.sqlite_metadata_connection_config(
          metadata_path
      ),
      enable_cache=True,
  )

## Eksekusi Otomatis Pipeline dengan Apache Beam Runner

Komponen yang telah dirangkai di atas dieksekusi secara otomatis dan berurutan menggunakan orchestrator `BeamDagRunner`. Seluruh artefak, metadata eksekusi, serta silsilah data (*data lineage*) dicatat secara terpusat pada basis data SQLite metadata store.

In [6]:
pipeline_instance = init_pipeline(
    pipeline_root=PIPELINE_ROOT,
    pipeline_name=PIPELINE_NAME,
    metadata_path=METADATA_PATH,
    data_root=DATA_ROOT,
    transform_module_file=TRANSFORM_MODULE_FILE,
    tuner_module_file=TUNER_MODULE_FILE,
    trainer_module_file=TRAINER_MODULE_FILE,
    serving_model_dir=SERVING_MODEL_DIR,
)

BeamDagRunner().run(pipeline_instance)

running bdist_wheel
running build
running build_py
creating build/lib
copying transform.py -> build/lib
copying trainer.py -> build/lib
copying tuner.py -> build/lib
installing to /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpuyxm2783
running install
running install_lib
copying build/lib/transform.py -> /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpuyxm2783/.
copying build/lib/trainer.py -> /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpuyxm2783/.
copying build/lib/tuner.py -> /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpuyxm2783/.
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing ma

/Users/erlanggajuni/anaconda3/envs/mlops-tfx-submission/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


running install_scripts
creating /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpuyxm2783/tfx_user_code_transform-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269.dist-info/WHEEL
creating '/var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmp_86hsrn_/tfx_user_code_transform-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269-py3-none-any.whl' and adding '/var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpuyxm2783' to it
adding 'trainer.py'
adding 'transform.py'
adding 'tuner.py'
adding 'tfx_user_code_transform-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269.dist-info/METADATA'
adding 'tfx_user_code_transform-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269.dist-info/WHEEL'
adding 'tfx_user_code_transform-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269.dist-info/top_level.txt'
adding 'tfx_user_code_transform-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269.dist-info

/Users/erlanggajuni/anaconda3/envs/mlops-tfx-submission/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


reading manifest file 'tfx_user_code_Tuner.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Tuner.egg-info/SOURCES.txt'
Copying tfx_user_code_Tuner.egg-info to /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmp3p32vi5b/./tfx_user_code_Tuner-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269-py3.10.egg-info
running install_scripts
creating /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmp3p32vi5b/tfx_user_code_tuner-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269.dist-info/WHEEL
creating '/var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpa82oxcef/tfx_user_code_tuner-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269-py3-none-any.whl' and adding '/var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmp3p32vi5b' to it
adding 'trainer.py'
adding 'transform.py'
adding 'tuner.py'
adding 'tfx_user_code_tuner-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269.dist-info/METADATA'
adding 'tfx_user_code_tuner-

/Users/erlanggajuni/anaconda3/envs/mlops-tfx-submission/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
Copying tfx_user_code_Trainer.egg-info to /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpd7x9yoo6/./tfx_user_code_Trainer-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269-py3.10.egg-info
running install_scripts
creating /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpd7x9yoo6/tfx_user_code_trainer-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269.dist-info/WHEEL
creating '/var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpxic04nwf/tfx_user_code_trainer-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269-py3-none-any.whl' and adding '/var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpd7x9yoo6' to it
adding 'trainer.py'
adding 'transform.py'
adding 'tuner.py'
adding 'tfx_user_code_trainer-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269.dist-info/METADATA'
adding 'tfx_user_code_trainer-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badeccc

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


## Verifikasi Model Hasil Ekspor Pusher

Setelah seluruh DAG pipeline selesai dieksekusi oleh Apache Beam, komponen Pusher akan mengekspor artefak model ke direktori serving jika model tersebut dinyatakan *Blessed* oleh Evaluator. Kode di bawah memverifikasi ketersediaan folder timestamp model serving hasil ekspor.

In [7]:
pushed_models = sorted(os.listdir(SERVING_MODEL_DIR))
print(f"Model yang berhasil diekspor di {SERVING_MODEL_DIR}:")
for m in pushed_models:
    print(" - Versi Timestamp:", m)

Model yang berhasil diekspor di erlanggajuni45-pipeline/serving_model:
 - Versi Timestamp: 1789208651
 - Versi Timestamp: 1789215343
 - Versi Timestamp: 1789311174
 - Versi Timestamp: 1789317220


## Menampilkan Evaluator

In [8]:
import glob
import pandas as pd

Mengambil path evaluasi dari eksekusi terakhir

In [9]:
eval_dirs = glob.glob(
    "erlanggajuni45-pipeline/pipeline_root/Evaluator/evaluation/*"
)
latest_eval_dir = max(eval_dirs, key=os.path.getmtime)
print(f"Memuat artefak Evaluator dari: {latest_eval_dir}")

Memuat artefak Evaluator dari: erlanggajuni45-pipeline/pipeline_root/Evaluator/evaluation/26


Memuat hasil evaluasi TFMA

In [10]:
eval_result = tfma.load_eval_result(latest_eval_dir)

Format dan tampilkan metrik slicing ke tabel

In [12]:
records = []
for slice_key, sub_dict in eval_result.slicing_metrics:
  metrics_data = sub_dict.get("", {}).get("", {})
  for metric_name, val_dict in metrics_data.items():
    val = (
        val_dict.get("doubleValue")
        if isinstance(val_dict, dict)
        else val_dict
    )
    records.append({"Metrik Evaluator (TFMA)": metric_name, "Nilai": val})

df_eval = pd.DataFrame(records)
display(df_eval)

,Metrik Evaluator (TFMA),Nilai
0,mean_absolute_error,1.298007e+05
1,mean_squared_error,1.504680e+11
2,mean_absolute_error_diff,0.000000e+00
3,mean_squared_error_diff,0.000000e+00


Verifikasi status blessing dari komponen Evaluator

In [13]:
blessing_dirs = glob.glob(
    "erlanggajuni45-pipeline/pipeline_root/Evaluator/blessing/*"
)
latest_blessing_dir = max(blessing_dirs, key=os.path.getmtime)
blessing_files = os.listdir(latest_blessing_dir)

print(f"Direktori Blessing: {latest_blessing_dir}")
print(f"Status Evaluator: {blessing_files}")
if "BLESSED" in blessing_files:
  print("Hasil Validasi: MODEL BLESSED (Lolos threshold dan layak di-push)")
else:
  print("Hasil Validasi: MODEL NOT BLESSED")

Direktori Blessing: erlanggajuni45-pipeline/pipeline_root/Evaluator/blessing/26
Status Evaluator: ['BLESSED']
Hasil Validasi: MODEL BLESSED (Lolos threshold dan layak di-push)
